# A declared value cannot be edited

Many objects may share EUR, so changing its label through one reference must fail.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field, ValidationError

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
try:
    CurrencyEnumEntity.EUR.display_name = "Changed"
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    raise AssertionError("The shared value was modified")
print(CurrencyEnumEntity.EUR.display_name)

frozen_instance
Euro


## What the result means

Assignment raises `frozen_instance`, and the shared label remains Euro. This demonstrates attribute protection. The separate dictionary-property annotation rule restricts nested types; frozen assignment alone is not a recursive audit of arbitrary objects.